In [1]:
import duckdb
import pandas as pd
import os

DATA_DIR = os.path.expanduser('~/kkbox-churn/data/')
con = duckdb.connect()

print("DuckDB version:", duckdb.__version__)
print("Data directory:", DATA_DIR)
print("Files found:")
for f in os.listdir(DATA_DIR):
    if f.endswith('.csv'):
        size = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024**3)
        print(f"  {f:<35} {size:.2f} GB")

DuckDB version: 1.5.3
Data directory: /Users/harshithnr/kkbox-churn/data/
Files found:
  sample_submission_zero.csv          0.04 GB
  user_logs_v2.csv                    1.33 GB
  user_logs.csv                       28.42 GB
  sample_submission_v2.csv            0.04 GB
  train_v2.csv                        0.04 GB
  transactions_v2.csv                 0.11 GB
  train.csv                           0.04 GB
  members_v3.csv                      0.40 GB
  transactions.csv                    1.61 GB


In [ ]:
files = {
    'train_v2'         : DATA_DIR + 'train_v2.csv',
    'members_v3'       : DATA_DIR + 'members_v3.csv',
    'transactions'     : DATA_DIR + 'transactions.csv',
    'transactions_v2'  : DATA_DIR + 'transactions_v2.csv',
    'user_logs_v2'     : DATA_DIR + 'user_logs_v2.csv',
}

for name, path in files.items():
    print(f"\n{'='*50}")
    print(f"TABLE: {name}")

    df = con.execute(f"SELECT * FROM read_csv_auto('{path}') LIMIT 3").df()
    print(df.dtypes)
    print(df.head(3))
    
    count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
    print(f"Total rows: {count:,}")


TABLE: train_v2
msno          str
is_churn    int64
dtype: object
                                           msno  is_churn
0  ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=         1
1  f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=         1
2  zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=         1
Total rows: 970,960

TABLE: members_v3
msno                         str
city                       int64
bd                         int64
gender                    object
registered_via             int64
registration_init_time     int64
dtype: object
                                           msno  city  bd gender  \
0  Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=     1   0   None   
1  +tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=     1   0   None   
2  cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=     1   0   None   

   registered_via  registration_init_time  
0              11                20110911  
1               7                20110914  
2              11                20110915  
T

In [3]:
print("=" * 50)
print("TABLE: user_logs (schema + count only)")

sample = con.execute(f"""
    SELECT * FROM read_csv_auto('{DATA_DIR}user_logs.csv')
    LIMIT 3
""").df()

print(sample.dtypes)
print(sample.head(3))

count = con.execute(f"""
    SELECT COUNT(*) FROM read_csv_auto('{DATA_DIR}user_logs.csv')
""").fetchone()[0]

print(f"Total rows: {count:,}")

TABLE: user_logs (schema + count only)
msno              str
date            int64
num_25          int64
num_50          int64
num_75          int64
num_985         int64
num_100         int64
num_unq         int64
total_secs    float64
dtype: object
                                           msno      date  num_25  num_50  \
0  rxIP2f2aN0rYNp+toI0Obt/N/FYQX8hcO1fTmmy2h34=  20150513       0       0   
1  rxIP2f2aN0rYNp+toI0Obt/N/FYQX8hcO1fTmmy2h34=  20150709       9       1   
2  yxiEWwE9VR5utpUecLxVdQ5B7NysUPfrNtGINaM2zA8=  20150105       3       3   

   num_75  num_985  num_100  num_unq  total_secs  
0       0        0        1        1     280.335  
1       0        0        7       11    1658.948  
2       0        0       68       36   17364.956  
Total rows: 392,106,543


In [4]:
tables = {
    'train_v2'        : DATA_DIR + 'train_v2.csv',
    'members_v3'      : DATA_DIR + 'members_v3.csv',
    'transactions'    : DATA_DIR + 'transactions.csv',
    'transactions_v2' : DATA_DIR + 'transactions_v2.csv',
    'user_logs_v2'    : DATA_DIR + 'user_logs_v2.csv',
}

for name, path in tables.items():
    print(f"\n{'='*50}")
    print(f"NULL ANALYSIS: {name}")
    df = con.execute(f"SELECT * FROM read_csv_auto('{path}') LIMIT 0").df()
    cols = df.columns.tolist()
    
    null_query = ", ".join([
        f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}_nulls"
        for c in cols
    ])
    
    result = con.execute(f"""
        SELECT {null_query}
        FROM read_csv_auto('{path}')
    """).df()
    
    print(result.T)


NULL ANALYSIS: train_v2
                  0
msno_nulls      0.0
is_churn_nulls  0.0

NULL ANALYSIS: members_v3
                                      0
msno_nulls                          0.0
city_nulls                          0.0
bd_nulls                            0.0
gender_nulls                  4429505.0
registered_via_nulls                0.0
registration_init_time_nulls        0.0

NULL ANALYSIS: transactions
                                0
msno_nulls                    0.0
payment_method_id_nulls       0.0
payment_plan_days_nulls       0.0
plan_list_price_nulls         0.0
actual_amount_paid_nulls      0.0
is_auto_renew_nulls           0.0
transaction_date_nulls        0.0
membership_expire_date_nulls  0.0
is_cancel_nulls               0.0

NULL ANALYSIS: transactions_v2
                                0
msno_nulls                    0.0
payment_method_id_nulls       0.0
payment_plan_days_nulls       0.0
plan_list_price_nulls         0.0
actual_amount_paid_nulls      0.0
is_

In [5]:
summary = {
    'table'     : ['train_v2', 'members_v3', 'transactions', 
                   'transactions_v2', 'user_logs_v2', 'user_logs'],
    'rows'      : [970960, 6769473, 21547746, 
                   1431009, 18396362, 392106543],
    'nulls'     : ['none', 'gender 65%', 'none', 
                   'none', 'none', 'not checked'],
    'join_key'  : ['msno'] * 6
}

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

con.close()
print("\nNotebook 01 complete.")

          table      rows       nulls join_key
       train_v2    970960        none     msno
     members_v3   6769473  gender 65%     msno
   transactions  21547746        none     msno
transactions_v2   1431009        none     msno
   user_logs_v2  18396362        none     msno
      user_logs 392106543 not checked     msno

Notebook 01 complete.
